# Basic workflow for making a "surrogate" model that learns the unpolarized BSA as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

## (1): Import Libraries

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split

## (2): Plotting Styles:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Data Loading:

In [ ]:
test_dataframe = pd.read_csv('../data/small_dset.csv')

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_target_bsa"]]

In [ ]:
x_data.head()

In [ ]:
y_data.head()

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

## (4): DNN Stuff:

### (4.1): MSE Loss:

In [ ]:
class BSALoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred))

### (4.2): DNN Architecture:

In [ ]:
class CrossSectionSurrogateModel(tf.keras.Model):

    # https://keras.io/api/models/model/ -> follow this for custom model architecture

    def __init__(self, symmetry_loss_weight = 1.0):
        super().__init__()

        self.symmetry_loss_weight = symmetry_loss_weight

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_2 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_3 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(1, activation = "linear", name = "cross_section")

        # custom loss business:
        self.cross_section_loss_tracker = BSALoss()

    def bsa_azimuthal_symmetry_loss(self, X_batch, training = True):

        X_plus = X_batch

        phi = X_batch[:, -1]

        X_minus = tf.concat([X_batch[:, :-1], tf.expand_dims(-phi, axis=1)], axis = 1)

        y_plus = self(X_plus, training = training)
        y_minus = self(X_minus, training = training)

        return tf.reduce_mean(tf.square(y_plus + y_minus))

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)
        cross_section_output = self.cross_section_output(hidden_layer)

        return cross_section_output
    
    def train_step(self, data):

        # unpack data:
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            # forward pass:
            predictions = self(X_batch_data, training = True)

            # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
            data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

            # compute BSA(phi) symmetry loss:
            symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = True)

            # total loss is just a weighted sum:
            total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.trainable_variables))

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

    def test_step(self, data):
        # unpack data:
        X_batch_data, y_batch_data = data
        
        # forward pass evaluation:
        predictions = self(X_batch_data, training = False)

        # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
        data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

         # compute BSA(phi) symmetry loss:
        symmetry_loss = self.bsa_azimuthal_symmetry_loss(X_batch_data, training = False)

        # total loss is just a weighted sum:
        total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)
            
        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

## (5): **Actually Fitting the Model**:

In [ ]:
NUMBER_OF_EPOCHS = 1250

tf.keras.backend.clear_session()

dnn_model = CrossSectionSurrogateModel(symmetry_loss_weight = 0.5)
dnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss = BSALoss())

dnn_model_history = dnn_model.fit(
    x_training, y_training,
    validation_data = (x_validation, y_validation),
    epochs = NUMBER_OF_EPOCHS, batch_size = len(x_training),
    verbose = 1)

number_of_epochs_run = len(dnn_model_history.epoch)
print(f"[INFO]: The model ran for {number_of_epochs_run} epochs before early stopping.")

In [ ]:
model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
model_testing_loss = dictionary_of_keras_metrics["loss"]

In [ ]:
figure, axis = plt.subplots(1, 1, figsize = (6, 6))

axis.plot(dnn_model_history.history['loss'], 
    label = "Training Loss", color = 'orange', alpha = 0.6)
axis.plot(dnn_model_history.history['val_loss'], 
    label = "Validation Loss", color = 'purple', alpha = 0.6)

axis.set_xlabel("Epoch", fontsize = 14.)
axis.set_ylabel("Loss Values", fontsize = 14.)
axis.set_title(rf"BSA Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
    fontsize = 14.)
axis.legend(fontsize = 14.)
axis.grid(visible = False)

axis.text(
    0.00, -0.11,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = axis.transAxes,
    verticalalignment = 'top',
    horizontalalignment = 'left',
    fontsize = 9.,
)

for extension in ['png', 'eps']:
    figure.savefig(
        fname = f"./plots/bsa_surrogate_test_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(figure)

In [ ]:
phi_smooth = np.linspace(-1.0*np.pi, 1.0*np.pi, 361)

# this is justified because the data we are using is *constant* in the kinematics
t_value = x_data["t"].mean()
xb_value = x_data["x_b"].mean()
qsquared_value = x_data["q_squared"].mean()

x_smooth = np.column_stack([
    np.full_like(phi_smooth, t_value),
    np.full_like(phi_smooth, xb_value),
    np.full_like(phi_smooth, qsquared_value),
    phi_smooth
])

predictions = dnn_model.predict(x_data).flatten()
actual = test_dataframe["unp_target_bsa"].values
phi_values = test_dataframe["phi"].values
residuals = actual - predictions

sum_squared_residuals = np.sum(residuals**2)
reduced_chi = sum_squared_residuals / len(phi_values)

residuals_figure, (ax1, ax2) = plt.subplots(2, 1, figsize = (10, 8), sharex = True, 
                               gridspec_kw = {'height_ratios': [3, 1]})

# Top Panel: Actual vs Predicted
ax1.scatter(phi_values, actual, color = 'black', label = 'Experimental Data', alpha = 0.7)
ax1.plot(phi_values, predictions, color = 'red', lw = 2, label = 'DNN Prediction')
ax1.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax1.set_ylabel(r"$\textrm{BSA}(\Lambda = 0)$", fontsize = 16.)
ax1.set_title(rf"BSA Surrogate Model (Testing = ${model_testing_loss:.3f}$)", fontsize = 14.)
ax1.legend()
ax1.grid(True, linestyle = ':', alpha = 0.6)

# Bottom Panel: Residuals
ax2.scatter(phi_values, residuals, color = 'blue', alpha = 0.6)
ax2.axhline(0, color = 'black', linestyle = '--')
ax2.set_ylabel(rf'Residuals ($\chi^2_\nu = {reduced_chi:.3f}$)', fontsize = 16.)
ax2.set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
ax2.grid(True, linestyle = ':', alpha = 0.6)

for extension in ['png', 'eps']:
    residuals_figure.savefig(
        fname = f"./plots/bsa_surrogate_residuals_v2.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(residuals_figure)